In [1]:
import os

In [3]:
from langchain.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_openai import OpenAI 

In [4]:
from langchain.document_loaders import DirectoryLoader
from langchain.document_loaders import TextLoader
from langchain_community.document_loaders import PyPDFLoader


In [5]:
texts=[]
for t in os.listdir("amu data"):
    if t==".DS_Store":
        continue
    print(t)
    path=f"amu data/{t}"
    loader=PyPDFLoader(path)
    text=loader.load()
    texts+=text

terms (1).pdf
phdregulation.pdf
schooladmguide_fn.pdf
HOSPITAL.pdf
PROCEDURE FOR NON-TEST COURSES (1).pdf
PROCEDURE FOR NON-TEST COURSES.pdf
ordinance_phd.pdf
terms.pdf
UFM_RULES-F2 (1).pdf
fn_supplement.pdf
txt doc amuict train data.pdf
guide to admission.pdf
PROCEDURE FOR DEPARTMENTAL TEST COURSES.pdf
termsand privacy policy.pdf
PROCEDURE FOR DEPARTMENTAL TEST COURSES (1).pdf
UFM_RULES-F2.pdf
LIST OF RECOGNIZED BOARDS OF EXAMINATION AND UNIVERSITIES.pdf
amu counselling guide (1).pdf
medicineguide.pdf
schooladmguide.pdf
6c79f5140da71180f8e833763617f2b4.pdf
e07cedbf89d39c7931a327df851484db.pdf
unaniguide.pdf
Examination (Control of Unfair Means and Disorderly Conduct) Regulations-2019.pdf
amu counselling guide.pdf


In [9]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
text=text_splitter.split_documents(text)
text[:200]


[Document(page_content='1 \n \n \n USER  MANUAL FOR ONLINE REGISTRATION, CHOICE FILLING \nAND AD MISSION \n(Session 202 4-25) \n \n• Admissions to the Aligarh  Muslim  University are be ing effected through the Online Counselling \n/Admission portal for admissions (https://couns elling.amuonline.ac.in). \n• All the eligible candidates who figure in the Select/Chance Memo/Nomination Lists are required to \nregister online as per notified schedule  of the concerned course on the Online portal for Counselling and \nAdmission from places of their convenience preferably using a Laptop/Desktop. \n• Candidates who do not complete all the steps leading to completion of admission as detailed below \nfor the course conc erned  as per notified schedule will be treated as candidates who are not interested \nin taking admission in the conc erned  course of study. HENCE , they will not be considered for \ncouns elling/admission to the con cerned  cour se and their allotted seat, if any, will be CANC

In [53]:
len(texts)

463

In [10]:
chunks=[document.page_content for document in texts]

# Contextual Chunk Headers (CCH)

In [36]:
from openai import OpenAI


In [ ]:
import tiktoken
from typing import List, Dict, Any, Tuple

DOCUMENT_TITLE_PROMPT = """
INSTRUCTIONS
What is the title of the following document?

Your response MUST be the title of the document, and nothing else. DO NOT respond with anything else.

{document_title_guidance}

{truncation_message}

DOCUMENT
{document_text}
""".strip()

TRUNCATION_MESSAGE = """
Also note that the document text provided below is just the first ~{num_words} words of the document. That should be plenty for this task. Your response should still pertain to the entire document, not just the text provided below.
""".strip()

MAX_CONTENT_TOKENS = 4000
MODEL_NAME = "gpt-4o-mini"
TOKEN_ENCODER = tiktoken.encoding_for_model('gpt-3.5-turbo')

def make_llm_call(chat_messages: list[dict]) -> str:
    """
    Make an API call to the OpenAI language model.

    Args:
        chat_messages (list[dict]): A list of message dictionaries for the chat completion.

    Returns:
        str: The generated response from the language model.
    """
    client = OpenAI(api_key="YOUR_OPENAI_API_KEY_HERE")
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=chat_messages,
        max_tokens=MAX_CONTENT_TOKENS,
        temperature=0.2,
    )
    return response.choices[0].message.content.strip()

def truncate_content(content: str, max_tokens: int) -> tuple[str, int]:
    """
    Truncate the content to a specified maximum number of tokens.

    Args:
        content (str): The input text to be truncated.
        max_tokens (int): The maximum number of tokens to keep.

    Returns:
        tuple[str, int]: A tuple containing the truncated content and the number of tokens.
    """
    tokens = TOKEN_ENCODER.encode(content, disallowed_special=())
    truncated_tokens = tokens[:max_tokens]
    return TOKEN_ENCODER.decode(truncated_tokens), min(len(tokens), max_tokens)

def get_document_title(chunks: List[str], document_title_guidance: str = "") -> str:
    """
    Extract the title of a document using a language model.

    Args:
        document_text (str): The text of the document.
        document_title_guidance (str, optional): Additional guidance for title extraction. Defaults to "".

    Returns:
        str: The extracted document title.
    """
    for document in chunks:
        document_text=document.page_content 
        # Truncate the content if it's too long
        document_text, num_tokens = truncate_content(document_text, MAX_CONTENT_TOKENS)
        truncation_message = TRUNCATION_MESSAGE.format(num_words=3000) if num_tokens >= MAX_CONTENT_TOKENS else ""
    
        # Prepare the prompt for title extraction
        prompt = DOCUMENT_TITLE_PROMPT.format(
            document_title_guidance=document_title_guidance,
            document_text=document_text,
            truncation_message=truncation_message
        )
        chat_messages = [{"role": "user", "content": prompt}]
        document_title = make_llm_call(chat_messages)
        chunk_w_header = f"Document Title: {document_title}\n\n{document_text}"
        document.page_content=chunk_w_header
    return chunks


if __name__ == "__main__":
    # Assuming document_text is defined elsewhere
    chunk_w_header = get_document_title(texts)
    # print(f"Document Title: {document_title}")

In [22]:
chunks=[document.page_content for document in chunk_w_header]

In [52]:
len(chunks)

463

## Question

In [41]:
from openai import OpenAI


client = OpenAI(api_key="YOUR_OPENAI_API_KEY_HERE")

def generate_questions(text_chunk):
    prompt = f"""
    You are a question generator.
    Your response MUST be the question , and nothing else. DO NOT respond with anything else.
    Given the following text, generate 20-30 meaningful, diverse questions.
    Questions should be relevant, not too repetitive, and cover different angles.
    
    Text:
    {text_chunk}
    
    Questions:
    """

    response = client.chat.completions.create(
        model="gpt-4o",  # or "gpt-4" or "gpt-3.5-turbo"
        messages=[
            {"role": "system", "content": "You generate questions from text."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.7,
        max_tokens=2000
    )

    # Correct way for new OpenAI library:
    questions = response.choices[0].message.content
    return questions



In [42]:
def get_document_Questions(chunks: List[str]) -> str:
    for document in chunks:
            document_text=document
            
            # Prepare the prompt for title extraction
        
            questions = generate_questions(document_text)
            chunk_w_header = f"{document_text}\n\n Document Questions: {questions}"
            document.page_content=chunk_w_header
    return chunks

In [54]:
chunks

['Document Title: Online Payments Terms & Conditions\n\nOnline Payments Terms & Conditions  \n \nThis online payment system is provided by Aligarh M uslim University Aligarh (AMU). The \nAMU may update these terms from time to time withou t any prior notice or reason and \nany changes will be effective immediately on being set out here. Please ensure you are \naware of the current terms. The country of domicile  for AMU is India. \nTerms & Conditions \nPlease read these terms carefully before using the online payment facility. Using the \nonline payment facility on this website indicates t hat you accept these terms. If you do \nnot accept these terms do not use this facility. AM U reserves the right to change the fees \nat any time. \nYour payment will normally reach AMU account to whi ch you are making a payment \nwithin two working days. \nWe cannot accept liability for a payment not reachi ng the correct AMU account if \npayment is refused or declined by the credit/debit card suppl

In [62]:
import json

In [63]:
with open('AMU_Documents_chunk.json', 'w', encoding='utf-8') as json_file:
    json.dump(chunks, json_file, indent=4, ensure_ascii=False)

print("Saved successfully as AMU_Documents.json!")

Saved successfully as AMU_Documents.json!


In [66]:
chunk_w_header

[Document(page_content="page_content='Document Title: Online Payments Terms & Conditions\\n\\nOnline Payments Terms & Conditions  \\n \\nThis online payment system is provided by Aligarh M uslim University Aligarh (AMU). The \\nAMU may update these terms from time to time withou t any prior notice or reason and \\nany changes will be effective immediately on being set out here. Please ensure you are \\naware of the current terms. The country of domicile  for AMU is India. \\nTerms & Conditions \\nPlease read these terms carefully before using the online payment facility. Using the \\nonline payment facility on this website indicates t hat you accept these terms. If you do \\nnot accept these terms do not use this facility. AM U reserves the right to change the fees \\nat any time. \\nYour payment will normally reach AMU account to whi ch you are making a payment \\nwithin two working days. \\nWe cannot accept liability for a payment not reachi ng the correct AMU account if \\npayment i

In [67]:
import pickle

# Assuming you have a list of Document objects
from langchain.schema import Document

with open('AMU_Documents_question.pkl', 'wb') as f:
    pickle.dump(chunk_w_header, f)

print("Documents saved successfully as AMU_Documents.pkl!")


Documents saved successfully as AMU_Documents.pkl!


In [68]:
# Load the documents
with open('AMU_Documents_question.pkl', 'rb') as f:
    loaded_documents = pickle.load(f)

print(f"Loaded {len(loaded_documents)} documents back!")

# Now loaded_documents is a list of Document objects — no need to reconstruct!


Loaded 463 documents back!


In [70]:
loaded_documents[0].page_content

"page_content='Document Title: Online Payments Terms & Conditions\\n\\nOnline Payments Terms & Conditions  \\n \\nThis online payment system is provided by Aligarh M uslim University Aligarh (AMU). The \\nAMU may update these terms from time to time withou t any prior notice or reason and \\nany changes will be effective immediately on being set out here. Please ensure you are \\naware of the current terms. The country of domicile  for AMU is India. \\nTerms & Conditions \\nPlease read these terms carefully before using the online payment facility. Using the \\nonline payment facility on this website indicates t hat you accept these terms. If you do \\nnot accept these terms do not use this facility. AM U reserves the right to change the fees \\nat any time. \\nYour payment will normally reach AMU account to whi ch you are making a payment \\nwithin two working days. \\nWe cannot accept liability for a payment not reachi ng the correct AMU account if \\npayment is refused or declined b

In [64]:
with open('AMU_Documents_chunk_with_question.json', 'w', encoding='utf-8') as json_file:
    json.dump(chunk_w_header, json_file, indent=4, ensure_ascii=False)

print("Saved successfully as AMU_Documents.json!")

TypeError: Object of type Document is not JSON serializable

In [51]:
print(len(chunk_w_header))

463


In [43]:
chunk_w_question = get_document_Questions(texts)

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

# chroma db

In [59]:
persist_directory='guideToAddmissinVectorheader_questions_and_title'

In [60]:
from langchain_openai import OpenAIEmbeddings
from langchain_openai import OpenAI 

In [1]:
embedding = OpenAIEmbeddings()

NameError: name 'OpenAIEmbeddings' is not defined

In [ ]:
vectordb=Chroma.from_documents(documents=chunk_w_header,
                               embedding=embedding,
                               persist_directory=persist_directory)

In [ ]:
vectordb.persist()
vectordb=None